[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Performance


## What you will be able to do

Time a loop against its array version with `timeit` and report the result fairly, explain from what a
list and an array store why the array is faster, and recognize the habits that make array code as
slow as a loop.

## The idea

### The problem

Every notebook in this guide has written a loop's work as one call on an array, and the **Why NumPy**
notebook measured the difference once, on a conversion to Fahrenheit. A claim of speed needs a
measurement behind it, and the measurement needs care, because a computer running other programs
makes any single timing noisy, and a timing on one machine says little about another.

Knowing why the array is faster matters for a second reason. Some code uses arrays and still runs at
the speed of a Python loop, or slower, and the reason it is slow follows from the same explanation.
Growing an array one value at a time, looping over an array in Python, and wrapping a function in
`np.vectorize` all look like NumPy code and do a Python loop's work.

### What vectorized means

> Code is **vectorized** when an operation on a whole array is written as one call, such as
> `bergen * 9 / 5 + 32` or `(bergen < 0).sum()`, so that the loop over the values runs inside NumPy's
> compiled code and not in Python.

### Why it works that way

A Python list holds references to objects. Each of Bergen's 8,760 readings in `readings_list` is a
separate float object of 24 bytes, reached through a reference of 8 bytes. For every value, a loop in
Python follows the reference, checks the object's type, finds the operation for that type, and makes a
new object to hold the result. An array holds the 8,760 numbers themselves, 8 bytes each, one after
another and all of one type, so NumPy's compiled loop does the arithmetic with none of that work for
each value. The list costs 32 bytes a value and the array 8.

The same account predicts what is slow. A Python loop over an array is slower than one over a list,
because every value read out of the array has to be wrapped in a new NumPy number first. `np.append`
copies the whole array every time it is called, so growing an array one value at a time copies it
once per value. `np.vectorize` accepts an array and calls a Python function once for every value in
it.

Timings differ between computers and between runs on the same computer. This notebook prints each one
as a comparison, "more than five times as long", with a margin well below the ratio that was
measured, so that the printed answer holds wherever the notebook runs, and it never prints seconds.

### Where this shows up

The columns of a table in the **Pandas, Deep Dive** guide are arrays, so the same rules decide which
table code is fast. A function applied row by row to a table runs in Python in the same way that
`np.vectorize` does. The timings in the **asyncpg and psycopg3, Deep Dive** guide are printed as
bands for the same reason this notebook prints comparisons.

### What this notebook covers

- timing two versions of a calculation with `timeit`, and printing the result as a comparison
- what a list stores for each value, and what an array stores
- memory per value with `itemsize` and `nbytes`, and the smaller dtypes
- calculations from this guide, timed as loops and as arrays
- three habits that make array code slow, in the Common errors

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import timeit
import numpy as np

values = list(range(100_000))
array = np.arange(100_000)
loop = min(timeit.repeat(lambda: [v * 2 for v in values], number=10, repeat=3))
vectorized = min(timeit.repeat(lambda: array * 2, number=10, repeat=3))
print("the loop took more than ten times as long:", loop > 10 * vectorized)
```

```
the loop took more than ten times as long: True
```

Doubling 100,000 numbers in a list comprehension took more than ten times as long as doubling them in
an array.

## Setup

Five imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions
- `timeit` times each version of a calculation
- `sys` reports how many bytes a Python object takes, with `sys.getsizeof`

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. This notebook times the
lists against the array. `hour_of_year(7, 1, 12)` gives the position of noon on 1 July along a
station's row.

In [1]:
import math
import sys
import timeit
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### Timing with timeit

`timeit.repeat(function, number=20, repeat=5)` runs a function 20 times, does that 5 times, and
returns the total time of each round. The smallest round is the best estimate of the function's own
cost, since the slower rounds were slowed by something else on the computer, and `best` keeps that smallest round:

In [2]:
bergen_list = readings_list[0]
bergen = readings[0]


def best(function, number=20):
    return min(timeit.repeat(function, number=number, repeat=5))


def frost_with_a_loop():
    return sum(1 for celsius in bergen_list if celsius < 0)


def frost_with_a_mask():
    return (bergen < 0).sum()


print(frost_with_a_loop(), frost_with_a_mask())
print("the loop took more than five times as long:", best(frost_with_a_loop) > 5 * best(frost_with_a_mask))

1220 1220
the loop took more than five times as long: True


Both versions count 1,220 frost hours, which is the first thing to check, because a timing compares
two versions only when they give the same answer. The loop took more than five times as long as the mask. The ratio itself changes from one
computer and one run to the next, so the cell prints only a comparison it can stand behind.

### What a list holds for each value

In [3]:
print(type(bergen_list[0]).__name__, type(bergen[0]).__name__)
print(len(set(map(id, bergen_list))))
print(sys.getsizeof(bergen_list[0]), sys.getsizeof([0.0] * HOURS))
print(bergen.itemsize, bergen.nbytes)

float float64
8760
24 70136
8 70080


The list's 8,760 values are 8,760 separate Python objects of 24 bytes apiece, as the count of distinct `id`s shows. The list itself takes 70,136 bytes, 8 for each reference and 56 for the list's
own header, so a reading in a list costs about 32 bytes. The array stores the numbers themselves, 8
bytes each, 70,080 bytes for the row, and reading one out of it makes a new `np.float64`.

### Smaller dtypes

An array's memory is its number of values times its `itemsize`, so a smaller dtype takes less, at the
cost of precision that the **dtype** notebook measured:

In [4]:
for dtype in [np.float64, np.float32, np.float16]:
    smaller = readings.astype(dtype)
    print(f"{np.dtype(dtype).name:8} {smaller.nbytes:7} bytes, changed by up to",
          np.nanmax(np.abs(smaller - readings)).round(4))

float64   280320 bytes, changed by up to 0.0
float32   140160 bytes, changed by up to 0.0
float16    70080 bytes, changed by up to 0.0063


`float32` halves the memory and changes no reading by as much as a ten-thousandth of a degree.
`float16` halves it again and moves some readings by more than half a hundredth of a degree, which is
still less than the tenth of a degree that the readings are rounded to.

### Calculations from this guide, as loops and as arrays

The coldest hour and the conversion to Fahrenheit, each checked for the same answer before it is
timed:

In [5]:
def coldest_with_a_loop():
    position = 0
    for hour, celsius in enumerate(bergen_list):
        if celsius < bergen_list[position]:
            position = hour
    return position


def fahrenheit_with_a_loop():
    return [celsius * 9 / 5 + 32 for celsius in bergen_list]


def fahrenheit_with_an_array():
    return bergen * 9 / 5 + 32


print(coldest_with_a_loop(), bergen.argmin())
print(np.array_equal(fahrenheit_with_a_loop(), fahrenheit_with_an_array()))
print("coldest hour, more than ten times as long:", best(coldest_with_a_loop) > 10 * best(bergen.argmin))
print("Fahrenheit, more than ten times as long:",
      best(fahrenheit_with_a_loop) > 10 * best(fahrenheit_with_an_array))

459 459
True
coldest hour, more than ten times as long: True
Fahrenheit, more than ten times as long: True


The loops found the same coldest hour, 459, and the same Fahrenheit values, value for value, because
both versions do the same arithmetic in the same order. The loop for the coldest hour makes a
comparison in Python for each of the 8,760 readings, where `argmin` makes the same comparisons in
compiled code, and the conversion's loop also makes a new float object for every result.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/14-performance-solutions.ipynb).

**1.** Time `np.nanmean(readings[2])` against a loop that keeps Svalbard's readings that are not `None` in `readings_list[2]` and divides their sum by their count, check that the two means agree to two places, and print whether the loop took more than twice as long.

In [6]:
# your code here

**2.** Make Bergen's readings as `int16` tenths of a degree, `np.round(bergen * 10).astype(np.int16)`, and print its `itemsize` and `nbytes` beside those of the `float64` row.

In [7]:
# your code here

**3.** Count Oslo's hours above 15 degrees with a loop over `readings_list[1]` and with a mask, check that the counts agree, and print whether the loop took more than five times as long.

In [8]:
# your code here

**4.** Build Tromso's readings in Fahrenheit by appending to a list in a loop and converting once with `np.array`, and check with `np.array_equal` that the result equals `readings[3] * 9 / 5 + 32`.

In [9]:
# your code here

**5.** Time `np.vectorize(math.sqrt)` against `np.sqrt` on the absolute values of Bergen's readings, and print whether `np.vectorize` took more than ten times as long.

In [10]:
# your code here

## Common errors

### No error, and a loop that grows an array with np.append

`np.append` makes a new array and leaves the one it was given as it was:

In [11]:
original = np.array([1.0, 2.0])
longer = np.append(original, 3.0)

print(original, longer, np.shares_memory(original, longer))

[1. 2.] [1. 2. 3.] False


So a loop that grows an array with `np.append` makes a new array, and copies everything so far into
it, on every pass:

In [12]:
def grow_with_append():
    grown = np.array([])
    for celsius in bergen_list:
        grown = np.append(grown, celsius * 9 / 5 + 32)
    return grown


def collect_in_a_list():
    collected = []
    for celsius in bergen_list:
        collected.append(celsius * 9 / 5 + 32)
    return np.array(collected)


print(np.array_equal(grow_with_append(), collect_in_a_list()))
print("np.append took more than three times as long:",
      best(grow_with_append, number=1) > 3 * best(collect_in_a_list, number=1))

True
np.append took more than three times as long: True


The loop copied 1 value, then 2, then 3, and so on up to 8,760, which is 38,373,180 copies in all. A
list grows in place, so collecting values in a list and converting once at the end is the way to build
an array whose length is not known in advance. When the calculation is arithmetic, as here,
`bergen * 9 / 5 + 32` needs no loop at all.

### No error, and a Python loop over an array, slower than over a list

In [13]:
def total_of_list():
    total = 0.0
    for celsius in bergen_list:
        total += celsius
    return total


def total_of_array():
    total = 0.0
    for celsius in bergen:
        total += celsius
    return total


print(round(total_of_list(), 1), round(total_of_array(), 1))
print("the loop over the array took more than 1.5 times as long:",
      best(total_of_array) > 1.5 * best(total_of_list))

70079.0 70079.0
the loop over the array took more than 1.5 times as long: True


Every pass of the loop over the array made a new `np.float64` to hold the value read, and adding it to
a Python float goes through NumPy's machinery for a single number, which is slower than Python's own
addition. When a loop is needed, `bergen.tolist()` turns the row into a list of Python floats first.
Without one, `bergen.sum()` gives the total in one call.

### No error, and np.vectorize, which is a loop

In [14]:
def heating_needed(day_mean):
    if day_mean < 15.5:
        return 15.5 - day_mean
    return 0.0


bergen_daily = bergen.reshape(365, 24).mean(axis=1)
heating_for_every_day = np.vectorize(heating_needed)

print(np.array_equal(heating_for_every_day(bergen_daily), np.maximum(15.5 - bergen_daily, 0.0)))
print("np.vectorize took more than five times as long:",
      best(lambda: heating_for_every_day(bergen_daily), number=50)
      > 5 * best(lambda: np.maximum(15.5 - bergen_daily, 0.0), number=50))

True
np.vectorize took more than five times as long: True


`np.vectorize` let `heating_needed`, written for one day, accept all 365, and it gave the same heating
degree days as `np.maximum` in the **Vectorized Math** notebook. It still calls the function once for
every day, in Python. NumPy's documentation says that `np.vectorize` "is provided primarily for
convenience, not for performance" and that "the implementation is essentially a for loop". Rewrite
the function with ufuncs and `np.where` when speed matters.

## Recap

- Time two versions with `timeit.repeat`, keep the smallest round, and check the answers agree first
- Print a timing as a comparison with a margin, never as seconds
- Explain the difference from what is stored, a Python object per value in a list and 8 bytes in an
  array
- Choose a smaller dtype to save memory when its precision is enough
- Avoid `np.append` in a loop, Python loops over arrays, and `np.vectorize` where a ufunc will do

## What is next

That is the end of this guide. You can build arrays and choose their dtype and shape, pick values by
position, by slice, by mask and by lists of positions, combine shapes by broadcasting, apply ufuncs
and reductions along any axis, draw reproducible random numbers, solve and fit with linear algebra,
and measure whether the result was worth it.

The **Pandas, Deep Dive** guide builds labeled tables on these arrays, where a column is an array with
a name and the rows have an index, and every rule from this guide about dtypes, `nan`, masks and loops
still applies. The library's front page lists every guide.

---

&#8592; **Previous:** [Linear Algebra](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/13-linear-algebra.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
